In [1]:
import polars as pl 
from dbconfig import engine 
print('Environment ready!')

Environment ready!


In [ ]:
import plotly.express as px

In [2]:
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_width_chars(180)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_float_precision(2)

polars.config.Config

In [6]:
pl.read_database(
        query = """
        select
        table_name
        from information_schema.tables
        where table_schema = 'analysis'
        order by table_name;
        """, connection = engine
        )

table_name
str
"""inflation"""
"""nifty_historical_fixed"""
"""nifty_historical_fixed_summary"""
"""nifty_historical_full"""
"""nifty_historical_full_summary"""
"""nifty_historical_half"""
"""nifty_historical_half_summary"""
"""nifty_monthly"""


In [10]:
pl.read_database(
        query = """
        select column_name
        from information_schema.columns 
        where table_name = 'nifty_historical_fixed_summary'
        """, connection = engine
        )

column_name
str
"""year"""
"""monthly_contribution"""
"""total_contributions"""
"""real_contributions"""
"""investment_gain"""
"""nominal_corpus"""
"""inflation_factor"""
"""real_corpus"""
"""real_wealth_created"""


In [12]:
fixed = pl.read_database(
        query = """
        select 
        year,
        total_contributions,
        nominal_corpus,
        real_corpus
        from analysis.nifty_historical_fixed_summary
        order by year;
        """, connection = engine
        )

In [13]:
half = pl.read_database(
        query = """
        select 
        year,
        total_contributions,
        nominal_corpus,
        real_corpus
        from analysis.nifty_historical_half_summary
        order by year;
        """, connection = engine
        )

In [14]:
full = pl.read_database(
        query = """
        select 
        year,
        total_contributions,
        nominal_corpus,
        real_corpus
        from analysis.nifty_historical_full_summary
        order by year;
        """, connection = engine
        )

In [15]:
fixed.tail()

year,total_contributions,nominal_corpus,real_corpus
i64,f64,f64,f64
2022,1625000.00,11953746.20,2048917.40
2023,1685000.00,14416821.85,2333425.78
2024,1745000.00,15746492.74,2414627.04
2025,1805000.00,17465052.36,2580113.20
2026,1810000.00,16929349.32,2432853.85


In [16]:
half.tail()

year,total_contributions,nominal_corpus,real_corpus
i64,f64,f64,f64
2022,2716389.69,16128646.72,2764511.18
2023,2868108.68,19533339.70,3161556.61
2024,3024037.86,21409921.47,3283078.73
2025,3182929.70,23828768.53,3520225.36
2026,3196356.06,23104476.66,3320258.45


In [17]:
full.tail()

year,total_contributions,nominal_corpus,real_corpus
i64,f64,f64,f64
2022,4739403.18,22522973.18,3860522.97
2023,5110106.77,27460139.47,4444543.89
2024,5501384.41,30271193.75,4641899.90
2025,5907530.60,33884099.66,5005700.02
2026,5942323.79,32869870.20,4723606.85


In [20]:
fixed = fixed.with_columns(
        pl.lit('fixed').alias('strategy')
        )

In [21]:
half = half.with_columns(
        pl.lit('half').alias('strategy')
        )

In [22]:
full = full.with_columns(
        pl.lit('full').alias('strategy')
        )

In [23]:
combined = pl.concat([fixed, half, full])

In [29]:
fig = px.line(
    combined,
    x="year",
    y="total_contributions",
    color="strategy",
    title="Growth in Total Contributions"
)
fig.write_html('plots/total_contributions.html')

In [30]:
fig = px.line(
    combined,
    x="year",
    y="nominal_corpus",
    color="strategy",
    title="Historical Nominal Wealth Growth"
)

fig.write_html('plots/nominal_wealth.html')

In [31]:
fig = px.line(
    combined,
    x="year",
    y="real_corpus",
    color="strategy",
    title="Historical Inflation-Adjusted Wealth Growth"
)

fig.write_html('plots/real_wealth.html')

In [39]:
crossover_fixed = pl.read_database(
        query = """
        select
        year,
        total_contributions,
        investment_gain
        from analysis.nifty_historical_fixed_summary
        order by year;
        """, connection = engine
        )

In [33]:
crossover_fixed.head()

year,total_contributions,investment_gain
i64,f64,f64
1995,5000.00,0.00
1996,65000.00,-4695.49
1997,125000.00,7932.79
1998,185000.00,-20132.32
1999,245000.00,105530.79


In [34]:
crossover_half = pl.read_database(
        query = """
        select
        year,
        total_contributions,
        investment_gain
        from analysis.nifty_historical_half_summary
        order by year;
        """, connection = engine
        )

In [35]:
crossover_full = pl.read_database(
        query = """
        select
        year,
        total_contributions,
        investment_gain
        from analysis.nifty_historical_full_summary
        order by year;
        """, connection = engine
        )

In [40]:
crossover_fixed = crossover_fixed.with_columns(
        (pl.col('investment_gain') > pl.col('total_contributions')).alias('market_gain_exceeds_investment')
        )

In [42]:
crossover_half = crossover_half.with_columns(
        (pl.col('investment_gain') > pl.col('total_contributions')).alias('market_gain_exceeds_investment')
        )

In [43]:
crossover_full = crossover_full.with_columns(
        (pl.col('investment_gain') > pl.col('total_contributions')).alias('market_gain_exceeds_investment')
        )

In [46]:
crossover_fixed.filter(pl.col('market_gain_exceeds_investment') == True).head(1)

year,total_contributions,investment_gain,market_gain_exceeds_investment
i64,f64,f64,bool
2005,605000.00,821970.16,true


In [47]:
crossover_half.filter(pl.col('market_gain_exceeds_investment') == True).head(1)

year,total_contributions,investment_gain,market_gain_exceeds_investment
i64,f64,f64,bool
2005,744551.78,986372.62,true


In [48]:
crossover_full.filter(pl.col('market_gain_exceeds_investment') == True).head(1)

year,total_contributions,investment_gain,market_gain_exceeds_investment
i64,f64,f64,bool
2005,914454.32,1182136.34,true


In [49]:
crossover_fixed = crossover_fixed.with_columns(
        pl.lit('fixed').alias('strategy')
        )

In [50]:
crossover_half = crossover_half.with_columns(
        pl.lit('half').alias('strategy')
        )

In [51]:
crossover_full = crossover_full.with_columns(
        pl.lit('full').alias('strategy')
        )

In [55]:
crossover_fixed.head(1)

year,total_contributions,investment_gain,market_gain_exceeds_investment,strategy
i64,f64,f64,bool,str
1995,5000.00,0.00,false,"""fixed"""


In [52]:
combined_ = pl.concat([crossover_fixed, crossover_half, crossover_full])

In [58]:
plot_df_ = (
        combined_.unpivot(
            index = ['year', 'strategy'],
            on = ['total_contributions', 'investment_gain'],
            variable_name = 'metric',
            value_name = 'value'
            )
        )


In [59]:
plot_df_ = plot_df_.with_columns(
    (
        pl.col("strategy") + " - " + pl.col("metric")
    ).alias("series")
)

In [62]:
plot_df_.head()

year,strategy,metric,value,series
i64,str,str,f64,str
1995,"""fixed""","""total_contributions""",5000.00,"""fixed - total_contributions"""
1996,"""fixed""","""total_contributions""",65000.00,"""fixed - total_contributions"""
1997,"""fixed""","""total_contributions""",125000.00,"""fixed - total_contributions"""
1998,"""fixed""","""total_contributions""",185000.00,"""fixed - total_contributions"""
1999,"""fixed""","""total_contributions""",245000.00,"""fixed - total_contributions"""


In [63]:
plot_df_.tail()

year,strategy,metric,value,series
i64,str,str,f64,str
2022,"""full""","""investment_gain""",17783570.00,"""full - investment_gain"""
2023,"""full""","""investment_gain""",22350032.70,"""full - investment_gain"""
2024,"""full""","""investment_gain""",24769809.35,"""full - investment_gain"""
2025,"""full""","""investment_gain""",27976569.07,"""full - investment_gain"""
2026,"""full""","""investment_gain""",26927546.41,"""full - investment_gain"""


In [61]:
fig = px.line(
    plot_df_,
    x="year",
    y="value",
    color="series",
    line_dash="metric",
    title="Contributions vs Investment Gain"
)

fig.write_html('plots/wealth_creation.html')

In [66]:
plot_df_.write_database(
    table_name="analysis.nifty_wealth_composition",
    connection=engine,
    if_table_exists="replace",
)

-1

In [64]:
plot_df_.schema

Schema([('year', Int64),
        ('strategy', String),
        ('metric', String),
        ('value', Float64),
        ('series', String)])